# 안심구역 분석 노트북 — 한전 EV 충전전력량 × 주유소 접근성 이중공백
- **파이썬 3 + pandas + numpy 만 필수** (scipy·matplotlib·statsmodels 는 있으면 추가 기능, 없으면 자동 생략)
- 한전 데이터: 날짜 · 시도 · 시군구 · 읍면동 · 고객호수 · 시간대(1:00~24:00), **사업자 / 비사업자 두 파일**
- 사용법: ① 셀 1 의 `PATH_KEPCO` 만 한전 파일 경로로 수정 → ② 위에서부터 순서대로 실행 → ③ `결과` 폴더 확인
- 윈도우 경로는 `r"D:\data\파일.csv"` 처럼 **앞에 r** 을 붙이세요 (안 붙이면 unicodeescape 에러)
- 각 셀은 중간 점검 결과를 출력합니다. 셀 6 의 **연결률**과 셀 7 의 **품질 점검**을 먼저 확인하세요.

In [ ]:
# [1] 설정 — 여기만 고치면 됩니다
import sys, os, re, io, csv, glob, json, codecs, unicodedata, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

# (1) 한전 파일 — 값: 파일 경로 하나 / 경로 목록 / 와일드카드(예: r"한전\비사업자_*.csv")
PATH_KEPCO = {"비사업자": "한전_비사업자.csv", "사업자": "한전_사업자.csv"}
DATA_DIR = "데이터"        # 패키지 데이터 폴더 (현재·상위 폴더도 자동 탐색)
OUT_DIR = "결과"           # 결과 저장 폴더 (자동 생성)

# (2) 자동 추정이 틀렸을 때만 지정 (None = 자동). 예: OVERRIDE["emd_col"] = "읍면동명"
OVERRIDE = dict(date_col=None, sido_col=None, sgg_col=None, emd_col=None, cust_col=None, hour_cols=None, code_col=None)

# (3) 분석 옵션
HOUR_LABEL_END = True      # 시간 컬럼이 1:00~24:00 처럼 '그 시각에 끝나는 1시간'이면 True (1:00 = 0~1시)
NIGHT_IDX = [23] + list(range(0, 9))   # 야간(경부하) 시간 0~23시 기준: 23시~다음날 9시. 한전 정의 확인 후 수정
UNIT_FACTOR = 1.0          # 값 단위가 Wh 면 0.001, MWh 면 1000
GAP_KMS = [3, 5, 7]        # 주유소 접근성 공백 기준(km) — 민감도 확인용
PRIMARY_GAP_KM = 5
CHARGE_QS = [0.10, 0.25, 0.50]   # '충전 실사용' 기준 = 인구 1천 명당 월 kWh 의 전국 분위수 — 민감도 확인용
PRIMARY_CHARGE_Q = 0.25
RUN_MONTHS = 3             # 기준을 연속 몇 개월 넘어야 '충전 시작'으로 볼지
GHOST_Q = 0.10             # 가동률 하위 분위수 = 유령 충전기 후보
MULTI_RULE = "rep"         # 한 법정동이 여러 행정동에 걸칠 때: "rep"(인구 최대 행정동 1곳) | "split"(인구 비례 분할)
MIN_CELL = 5               # 반출 후보 표에서 이 값 미만의 건수는 가림 (반출 규정에 맞게 조정)
DROP_PARTIAL_LAST_MONTH = True   # 마지막 달 자료가 직전 달의 90% 미만이면 불완전한 달로 보고 분석에서 제외
CHUNK = 500000             # 큰 파일을 읽는 단위(행). 메모리가 부족하면 줄이세요

_pv = tuple(int(x) for x in re.findall(r"\d+", pd.__version__)[:2])
if _pv < (0, 25):
    raise RuntimeError("pandas %s 는 너무 오래된 버전입니다(0.25 이상 필요). 가능하면 1.0 이상을 쓰세요." % pd.__version__)
try:
    for sub in ["내부용", "반출후보", "그림"]:
        os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
except OSError as e_:
    raise RuntimeError("결과 폴더를 만들 수 없습니다(%s). 쓰기 가능한 위치의 폴더로 OUT_DIR 를 바꾸세요. 예: OUT_DIR = r\"D:\\작업\\결과\"" % e_)
OPT = {}
for m_ in ["scipy", "matplotlib", "statsmodels"]:
    try:
        __import__(m_); OPT[m_] = True
    except Exception:
        OPT[m_] = False
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| 선택 라이브러리:", OPT)
LOG = {}       # 실행 요약 (마지막 셀에서 저장)

In [ ]:
# [2] 도구 함수 (수정 불필요)
ALIAS_SIDO = {"서울": "서울특별시", "서울시": "서울특별시", "부산": "부산광역시", "부산시": "부산광역시", "대구": "대구광역시", "대구시": "대구광역시",
              "인천": "인천광역시", "인천시": "인천광역시", "광주": "광주광역시", "대전": "대전광역시", "대전시": "대전광역시", "울산": "울산광역시", "울산시": "울산광역시",
              "세종": "세종특별자치시", "세종시": "세종특별자치시", "경기": "경기도", "강원": "강원특별자치도", "강원도": "강원특별자치도",
              "충북": "충청북도", "충남": "충청남도", "전북": "전북특별자치도", "전라북도": "전북특별자치도", "전남": "전라남도",
              "경북": "경상북도", "경남": "경상남도", "제주": "제주특별자치도", "제주도": "제주특별자치도"}

def norm(x):
    s = unicodedata.normalize("NFKC", str(x))
    s = re.sub(r"\([^)]*\)", "", s)
    return re.sub(r"\s+", "", s)

def norm_sido(x):
    n = norm(x)
    return ALIAS_SIDO.get(n, n)

def norm_sgg(x):
    n = norm(x)
    return {"세종시": "세종특별자치시"}.get(n, n)

def strip_gu(g):
    return re.sub(r"시[가-힣]+구$", "시", g)          # 수원시장안구 → 수원시

def strip_num(e):
    return re.sub(r"\d+(?=동$)", "", e)                # 가정1동 → 가정동

def find_file(name):
    if os.path.isabs(name) and os.path.exists(name):
        return name
    for base in [".", DATA_DIR, os.path.join("..", DATA_DIR), "..", "데이터", os.path.join("..", "데이터")]:
        p = os.path.join(base, name)
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join("**", name), recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError("파일을 찾을 수 없습니다: %s\n  현재 폴더: %s\n  → 패키지의 '데이터' 폴더가 노트북(코드 폴더) 옆에 그대로 있는지, 한전 파일이라면 셀 1 의 PATH_KEPCO 가 실제 경로인지 확인하세요.\n  → 현재 폴더 파일: %s" % (name, os.getcwd(), sorted(os.listdir("."))[:30]))

def expand_paths(spec):
    items = [spec] if isinstance(spec, str) else list(spec)
    out = []
    for it in items:
        if any(ch in it for ch in "*?["):
            out += sorted(glob.glob(it))
        elif os.path.exists(it):
            out.append(it)
        else:
            out.append(find_file(it))
    if not out:
        raise FileNotFoundError("한전 파일 경로에 해당하는 파일이 없습니다: %s\n  → 셀 1 의 PATH_KEPCO 를 한전 파일의 실제 경로로 고치세요. 윈도우 경로는 앞에 r 을 붙입니다. 예: r\"D:\\한전\\비사업자.csv\"\n  → 현재 폴더: %s" % (spec, os.getcwd()))
    return out

def sniff_encoding(path):
    with open(path, "rb") as f:
        raw = f.read(300000)
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return "utf-16"
    for enc in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            codecs.getincrementaldecoder(enc)("strict").decode(raw, final=False)
            return enc
        except UnicodeDecodeError:
            continue
    return "cp949"

HDR_KW = re.compile(r"시도|시군구|읍면동|법정동|행정동|코드|일자|날짜|연월|고객|호수|구분|kwh|사용량|\d{1,2}\s*[:시]", re.I)

def find_header_row(rows, max_scan=30):
    best, bi = -1, 0
    for i, r in enumerate(rows[:max_scan]):
        cells = [str(c).strip() for c in r]
        if sum(1 for c in cells if c) < 3:
            continue
        score = sum(1 for c in cells if HDR_KW.search(c)) - sum(1 for c in cells if re.fullmatch(r"-?[\d,\.]+", c))
        if score > best:
            best, bi = score, i
    return bi

def file_format(path):
    """인코딩·구분자·헤더 행 위치를 자동 판별"""
    enc = sniff_encoding(path)
    with open(path, "r", encoding=enc, errors="replace") as f:
        lines = [next(f, "") for _ in range(40)]
    lines = [l for l in lines if l.strip()]
    best, sep = -1, ","
    for d in [",", "\t", ";", "|"]:
        cnt = [l.count(d) for l in lines]
        sc = min(cnt) if cnt else 0
        if sc > best:
            best, sep = sc, d
    rows = list(csv.reader(lines, delimiter=sep))
    return enc, sep, find_header_row(rows)

def clean_columns(df):
    cols, seen = [], {}
    for c in df.columns:
        c = re.sub(r"\s+", "", str(c).replace("\ufeff", ""))
        seen[c] = seen.get(c, 0) + 1
        cols.append(c if seen[c] == 1 else "%s_%d" % (c, seen[c]))
    df.columns = cols
    return df

def read_csv_kw(path, **extra):
    enc, sep, hdr = file_format(path)
    kw = dict(encoding=enc, sep=sep, skiprows=hdr, header=0, dtype=str, low_memory=False)
    kw.update(extra)
    try:
        return pd.read_csv(path, on_bad_lines="skip", **kw)
    except TypeError:
        return pd.read_csv(path, error_bad_lines=False, warn_bad_lines=False, **kw)

def read_excel_all(path, **kw):
    try:
        return pd.read_excel(path, sheet_name=None, header=None, dtype=str, **kw)
    except ImportError as e_:
        raise ImportError("엑셀 파일을 읽는 라이브러리가 없습니다(%s). 한전 파일을 CSV 로 내보내거나(권장), 안심구역에서 가능하면 openpyxl(.xlsx) / xlrd(.xls) 를 설치하세요." % e_)

def read_preview(path, nrows=3000):
    if path.lower().endswith((".xlsx", ".xls", ".xlsm")):
        sheets = read_excel_all(path, nrows=nrows + 40)
        raw = list(sheets.values())[0]
        hdr = find_header_row(raw.fillna("").values.tolist())
        df = raw.iloc[hdr + 1:].copy()
        df.columns = raw.iloc[hdr].fillna("").tolist()
        return clean_columns(df.reset_index(drop=True))
    return clean_columns(read_csv_kw(path, nrows=nrows))

def iter_chunks(path, chunk):
    """파일을 청크 단위(DataFrame, 전부 문자열)로 읽는다. 엑셀은 시트별 1개 청크."""
    if path.lower().endswith((".xlsx", ".xls", ".xlsm")):
        for name, raw in read_excel_all(path).items():
            hdr = find_header_row(raw.fillna("").values.tolist())
            df = raw.iloc[hdr + 1:].copy()
            df.columns = raw.iloc[hdr].fillna("").tolist()
            yield clean_columns(df.reset_index(drop=True))
        return
    for ch in read_csv_kw(path, chunksize=chunk):
        yield clean_columns(ch)

HOUR_RE = re.compile(r"^(?:kwh|h|hr|hour|t|시간)?_?(\d{1,2})(?::\d{2}(?::\d{2})?|시|h|hr|시간)?(?:~\d{1,2}(?::\d{2})?(?:시)?)?$", re.I)

def parse_hour_cols(cols):
    found = []
    for c in cols:
        m = HOUR_RE.match(str(c))
        if m and 0 <= int(m.group(1)) <= 24:
            found.append((int(m.group(1)), c))
    if len(set(h for h, _ in found)) >= 20:
        return sorted(found, key=lambda x: x[0])
    return []

def hour_index(label, labels):
    if 0 in labels:
        return label % 24
    return (label - 1) % 24 if HOUR_LABEL_END else label % 24

def parse_month(s):
    """다양한 날짜 표기(20230115 / 2023-01-15 / 2023.01 / 202301 / 엑셀 일련번호 / 날짜시간) → 월 첫날. 실패 = NaT"""
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    ext = s.str.extract(r"((?:19|20)\d{2})\D*(\d{1,2})")
    y = pd.to_numeric(ext[0], errors="coerce")
    m = pd.to_numeric(ext[1], errors="coerce")
    ok = y.notna() & m.between(1, 12)
    if ok.any():
        out.loc[ok] = pd.to_datetime(pd.DataFrame({"year": y[ok].astype(int), "month": m[ok].astype(int), "day": 1}))
    rest = ~ok
    if rest.any():
        dig = s[rest].str.replace(r"\.0$", "", regex=True)
        ser = dig.str.match(r"^\d{5}$")
        if ser.any():
            v = pd.to_datetime(pd.to_numeric(dig[ser]), unit="D", origin="1899-12-30", errors="coerce")
            out.loc[ser[ser].index] = v.dt.to_period("M").dt.to_timestamp()
        rest2 = out.isna() & rest
        if rest2.any():
            v = pd.to_datetime(s[rest2], errors="coerce")
            out.loc[rest2] = v.dt.to_period("M").dt.to_timestamp()
    return out

def to_num_frame(df):
    return df.apply(lambda c: pd.to_numeric(c.astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce"))

def to_num_series(s):
    return pd.to_numeric(s.astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce")

def months_between(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)

print("도구 함수 준비 완료")

In [ ]:
# [3] 한전 파일 구조 확인 + 컬럼 자동 추정
def detect_schema(df):
    cols = list(df.columns)
    hour = parse_hour_cols(cols)
    hcols = set(c for _, c in hour)
    sc = dict(hour=hour, year_col=None, month_col=None)

    def pick(pred):
        for c in cols:
            if c not in hcols and pred(str(c)):
                return c
        return None

    sc["date_col"] = pick(lambda c: re.search(r"일자|날짜|일시|date|기준일|사용일|연월|기준월|사용월|^ym$", c, re.I) is not None)
    if not sc["date_col"]:
        sc["year_col"] = pick(lambda c: re.search(r"^(연도|년도|년|year|yyyy)$", c, re.I) is not None)
        sc["month_col"] = pick(lambda c: re.search(r"^(월|month|mm)$", c, re.I) is not None)
    sc["cust_col"] = pick(lambda c: re.search(r"고객|호수|계약", c) is not None)
    kc = [c for c in cols if c not in hcols and re.search(r"kwh|사용량|전력량|충전량", c, re.I)]
    tot = [c for c in kc if re.search(r"합계|총|계$", c)]
    sc["kwh_cols"] = tot if tot else kc

    reg = dict(sido=None, sgg=None, emd=None, sido_code=None, sgg_code=None, emd_code=None)
    for c in cols:
        if c in hcols:
            continue
        s = str(c)
        is_code = re.search(r"코드|code|_cd|cd$", s, re.I) is not None
        if "시도" in s:
            lv = "sido"
        elif "시군구" in s or "구군" in s:
            lv = "sgg"
        elif any(k in s for k in ["읍면동", "법정동", "행정동", "동명", "읍면"]):
            lv = "emd"
        else:
            continue
        key = lv + "_code" if is_code else lv
        if reg[key] is None:
            reg[key] = c
    for lv in ["sido", "sgg", "emd"]:      # 이름 컬럼인데 값이 숫자면 코드로 재분류
        c = reg[lv]
        if c is not None:
            smp = df[c].dropna().head(2000).astype(str).str.replace(r"\.0$", "", regex=True)
            if len(smp) and smp.str.match(r"^\d+$").mean() > 0.95:
                reg[lv + "_code"], reg[lv] = c, None
    sc.update(reg)
    return sc

def code_parts_from(df, sc):
    """코드 컬럼으로 법정동 코드를 이어붙이는 방법 [(컬럼, 자릿수), ...] (없으면 [])"""
    def width(c):
        return int(df[c].dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.len().max())
    if OVERRIDE.get("code_col"):
        return [(OVERRIDE["code_col"], width(OVERRIDE["code_col"]))]
    e, g, s_ = sc["emd_code"], sc["sgg_code"], sc["sido_code"]
    if e:
        we = width(e)
        if we >= 8:
            return [(e, we)]
        if g and width(g) >= 5:
            return [(g, 5), (e, we)]
        if g and s_:
            return [(s_, 2), (g, 3), (e, we)]
    return []

def apply_override(sc, df):
    for k, kk in [("date_col", "date_col"), ("sido_col", "sido"), ("sgg_col", "sgg"), ("emd_col", "emd"), ("cust_col", "cust_col")]:
        if OVERRIDE.get(k):
            sc[kk] = OVERRIDE[k]
    if OVERRIDE.get("hour_cols"):
        sc["hour"] = parse_hour_cols(OVERRIDE["hour_cols"]) or [(i, c) for i, c in enumerate(OVERRIDE["hour_cols"])]
    sc["code_parts"] = code_parts_from(df, sc)
    return sc

def validate_schema(sc, path):
    errs = []
    if not (sc["date_col"] or (sc["year_col"] and sc["month_col"])):
        errs.append("날짜 컬럼을 못 찾음 → OVERRIDE['date_col']")
    if not (sc["hour"] or sc["kwh_cols"]):
        errs.append("시간대(1:00~24:00) 또는 kWh 컬럼을 못 찾음 → OVERRIDE['hour_cols']")
    if not (sc["code_parts"] or sc["emd"]):
        errs.append("지역 컬럼(시도·시군구·읍면동 또는 코드)을 못 찾음 → OVERRIDE['sido_col'/'sgg_col'/'emd_col']")
    if errs:
        raise ValueError("%s: %s | 현재 컬럼: %s" % (os.path.basename(path), " / ".join(errs), list(sc.get("_cols", []))[:40]))

FILES = {}        # kind → 파일 목록
SCHEMAS = {}      # 파일 → 스키마
for kind, spec in PATH_KEPCO.items():
    FILES[kind] = expand_paths(spec)
    for p in FILES[kind]:
        prev = read_preview(p)
        sc = detect_schema(prev)
        sc["_cols"] = list(prev.columns)
        sc = apply_override(sc, prev)
        SCHEMAS[p] = sc
        print("=" * 78)
        print("[%s] %s  (미리보기 %d행)" % (kind, p, len(prev)))
        print("   컬럼:", list(prev.columns)[:12], "..." if prev.shape[1] > 12 else "")
        mode = "코드 이어붙이기 %s" % sc["code_parts"] if sc["code_parts"] else "지역 이름으로 연결"
        print("   지역: 시도=%s 시군구=%s 읍면동=%s | 방식: %s" % (sc["sido"], sc["sgg"], sc["emd"], mode))
        print("   날짜: %s | 고객호수: %s | 시간대 컬럼 %d개 %s" % (sc["date_col"] or (sc["year_col"], sc["month_col"]), sc["cust_col"], len(sc["hour"]),
              [c for _, c in sc["hour"]][:3] + ["..."] if sc["hour"] else ("kWh 컬럼: %s" % sc["kwh_cols"])))
        rc = [c for c in [sc["sido"], sc["sgg"], sc["emd"]] if c]
        if rc:
            print("   지역 값 예시:", prev[rc].drop_duplicates().head(3).values.tolist())
        validate_schema(sc, p)
print("\n※ 위 추정이 틀리면 셀 1 의 OVERRIDE 에 직접 지정한 뒤 셀 3 부터 다시 실행하세요.")

In [ ]:
# [4] 한전 데이터 → 월별 집계 (큰 파일은 청크 단위로 처리)
JUNK = {"합계", "소계", "총계", "계", "전체", "total", "Total", "TOTAL"}

def build_chunk(ch, sc, stats):
    stats["rows"] += len(ch)
    names = [c for c in [sc["sido"], sc["sgg"], sc["emd"]] if c]
    if names:
        junk = ch[names].apply(lambda s: s.astype(str).str.strip().isin(JUNK)).any(axis=1)
        allnull = ch[names].isna().all(axis=1)
        keep = ~(junk | allnull)
        stats["dropped_rows"] += int((~keep).sum())
        ch = ch[keep]
    if len(ch) == 0:
        return None
    if sc["code_parts"]:
        parts = [ch[c].astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(w) for c, w in sc["code_parts"]]
        code = parts[0]
        for p_ in parts[1:]:
            code = code + p_
        code = code.where(code.str.match(r"^\d+$"), "")
    else:
        code = pd.Series("", index=ch.index)
    if names:
        nm = [ch[sc[k]].fillna("").astype(str) if sc[k] else pd.Series("", index=ch.index) for k in ("sido", "sgg", "emd")]
        name = nm[0] + "\t" + nm[1] + "\t" + nm[2]
    else:
        name = pd.Series("", index=ch.index)
    if sc["date_col"]:
        month = parse_month(ch[sc["date_col"]])
    else:
        yy = to_num_series(ch[sc["year_col"]])
        yy = yy.where(yy >= 100, yy + 2000)
        mm = to_num_series(ch[sc["month_col"]])
        ok_ = yy.notna() & mm.between(1, 12)
        month = pd.Series(pd.NaT, index=ch.index, dtype="datetime64[ns]")
        month.loc[ok_] = pd.to_datetime(pd.DataFrame({"year": yy[ok_].astype(int), "month": mm[ok_].astype(int), "day": 1}))
    if sc["hour"]:
        hc = [c for _, c in sc["hour"]]
        raw = ch[hc]
        H = to_num_frame(raw)
        stats["masked"] += int((H.isna() & raw.notna()).values.sum())
        stats["cells"] += int(raw.notna().values.sum())
        labels = [h for h, _ in sc["hour"]]
        nc = [c for (h, c) in sc["hour"] if hour_index(h, labels) in NIGHT_IDX]
        kwh = H.sum(axis=1, min_count=1) * UNIT_FACTOR
        night = H[nc].sum(axis=1, min_count=1) * UNIT_FACTOR
        stats["neg"] += int((H < 0).values.sum())
        stats["zero"] += int((H == 0).values.sum())
    else:
        V = to_num_frame(ch[sc["kwh_cols"]])
        kwh = V.sum(axis=1, min_count=1) * UNIT_FACTOR
        night = pd.Series(np.nan, index=ch.index)
        stats["neg"] += int((V < 0).values.sum())
        stats["zero"] += int((V == 0).values.sum())
        stats["cells"] += int(V.notna().values.sum())
    cust = to_num_series(ch[sc["cust_col"]]) if sc["cust_col"] else pd.Series(np.nan, index=ch.index)
    d = pd.DataFrame({"code": code, "name": name, "month": month, "kwh": kwh, "night": night, "cust": cust})
    stats["bad_month"] += int(d["month"].isna().sum())
    d = d[d["month"].notna()]
    if len(d) == 0:
        return None
    return d.groupby(["code", "name", "month"]).agg(kwh=("kwh", "sum"), night=("night", "sum"), cust=("cust", "max"), rows=("kwh", "size")).reset_index()

parts_all, STATS = [], {}
for kind, plist in FILES.items():
    for p in plist:
        sc = SCHEMAS[p]
        st_ = dict(rows=0, dropped_rows=0, masked=0, cells=0, neg=0, zero=0, bad_month=0)
        parts = []
        for ch in iter_chunks(p, CHUNK):
            if not all(c in ch.columns for c in ([sc["date_col"]] if sc["date_col"] else []) + [c for _, c in sc["hour"]]):
                ch_cols = list(ch.columns)
                raise ValueError("%s: 청크의 컬럼이 미리보기와 다릅니다(파일 중간에 헤더가 바뀜?): %s" % (os.path.basename(p), ch_cols[:12]))
            r = build_chunk(ch, sc, st_)
            if r is not None:
                parts.append(r)
        if not parts:
            raise ValueError("%s: 읽을 수 있는 행이 없습니다(날짜 형식 또는 컬럼 확인)" % os.path.basename(p))
        agg = pd.concat(parts).groupby(["code", "name", "month"], as_index=False).agg(kwh=("kwh", "sum"), night=("night", "sum"), cust=("cust", "max"), rows=("rows", "sum"))
        agg["kind"] = kind
        parts_all.append(agg)
        STATS[p] = st_
        print("[%s] %s: 읽은 행 %s, 제외(합계 등) %s, 날짜 해석 실패 %s, 숫자 아님(마스킹 등) %s / 값 %s, 0값 %s, 음수 %s" % (
            kind, os.path.basename(p), format(st_["rows"], ","), st_["dropped_rows"], st_["bad_month"], st_["masked"], format(st_["cells"], ","), st_["zero"], st_["neg"]))
M = pd.concat(parts_all, ignore_index=True)
M = M.groupby(["code", "name", "month", "kind"], as_index=False).agg(kwh=("kwh", "sum"), night=("night", "sum"), cust=("cust", "max"), rows=("rows", "sum"))
DATA_START, DATA_END = M["month"].min(), M["month"].max()
DATA_MONTHS = pd.date_range(DATA_START, DATA_END, freq="MS")
print("-" * 78)
print(M.groupby("kind").agg(지역키수=("name", "nunique"), 월수=("month", "nunique"), 시작=("month", "min"), 끝=("month", "max"), kWh합계=("kwh", "sum")))
miss_m = [m_ for m_ in DATA_MONTHS if m_ not in set(M["month"])]
print("한전 데이터 기간: %s ~ %s (%d개월)%s" % (DATA_START.strftime("%Y-%m"), DATA_END.strftime("%Y-%m"), len(DATA_MONTHS), " | 빠진 달: %s" % [x.strftime("%Y-%m") for x in miss_m] if miss_m else ""))
last = M[M["month"] == DATA_END]["rows"].sum()
prev_ = M[M["month"] == DATA_END - pd.DateOffset(months=1)]["rows"].sum()
if prev_ and last < 0.9 * prev_:
    print("※ 마지막 달(%s) 자료 행 수가 직전 달의 %.0f%% 입니다 → 불완전한 달일 수 있음(셀 7 에서 제외 옵션)" % (DATA_END.strftime("%Y-%m"), 100 * last / prev_))
if DROP_PARTIAL_LAST_MONTH and prev_ and last < 0.9 * prev_:
    M = M[M["month"] < DATA_END].copy()
    DATA_END = M["month"].max()
    DATA_MONTHS = pd.date_range(DATA_START, DATA_END, freq="MS")
    print("   → 불완전한 마지막 달을 제외했습니다. 분석 기간: %s ~ %s" % (DATA_START.strftime("%Y-%m"), DATA_END.strftime("%Y-%m")))
LOG["한전기간"] = [DATA_START.strftime("%Y-%m"), DATA_END.strftime("%Y-%m")]
LOG["한전_행수"] = {os.path.basename(k): v["rows"] for k, v in STATS.items()}

In [ ]:
# [5] 패키지 데이터 읽기 (행정동 마스터 · 월별 주유소 거리 · 법정동 연결표)
def read_pkg(name, dtype=None, required=True):
    try:
        p = find_file(name)
    except FileNotFoundError:
        if required:
            raise
        print("  (선택 파일 없음) %s" % name)
        return None
    return pd.read_csv(p, encoding=sniff_encoding(p), dtype=dtype, low_memory=False)

master = read_pkg("행정동_마스터.csv", dtype={"행정동코드": str, "시군구코드": str})
need = ["행정동코드", "행정동명", "시도", "시군구코드", "시군구명", "인구", "65세이상", "고령비율", "인구감소지역", "면적km2", "충전기개소수_현재", "충전기거리_현재"]
miss = [c for c in need if c not in master.columns]
if miss:
    raise ValueError("행정동_마스터.csv 에 없는 컬럼: %s" % miss)
dong = master.set_index("행정동코드")
for c in ["인구", "65세이상", "고령비율", "면적km2", "충전기개소수_2021", "충전기개소수_현재", "충전기거리_현재", "주유소개소수_현재"]:
    if c in dong.columns:
        dong[c] = pd.to_numeric(dong[c], errors="coerce")
Dm = read_pkg("행정동_월별_최근접주유소거리.csv", dtype={"행정동코드": str}).set_index("행정동코드").reindex(dong.index)
Nm = read_pkg("행정동_월별_경계안주유소수.csv", dtype={"행정동코드": str}).set_index("행정동코드").reindex(dong.index)
GAS_MONTHS = pd.to_datetime([c + "-01" for c in Dm.columns])
D_ = Dm.values.astype(float)
N_ = Nm.values.astype(float)
sggm = read_pkg("시군구_마스터.csv", dtype={"시군구코드": str})
bridge = read_pkg("외부데이터_병합_법정동기준_반입용.csv", dtype=str, required=False)
xw = read_pkg("법정동_행정동_연계표_최신(2025-04-01).csv", dtype=str, required=False)
print("행정동 %d개 | 주유소 월별 거리 %s ~ %s | 법정동 연결 파일: %s | 연계표: %s" % (len(dong), GAS_MONTHS[0].strftime("%Y-%m"), GAS_MONTHS[-1].strftime("%Y-%m"),
      "있음" if bridge is not None else "없음", "있음" if xw is not None else "없음"))

In [ ]:
# [6] 한전 지역 → 우리 행정동 연결 (코드·이름·행정동/법정동 모든 경우) + 연결률 보고
K = M.groupby(["code", "name"], as_index=False)["kwh"].sum()
K["kwh"] = K["kwh"].fillna(0.0)
K["uid"] = np.arange(len(K))
K["bjd"] = None
K["adm"] = None
K["how"] = ""
sp_ = K["name"].str.split("\t", expand=True)
for i_ in range(3):
    if i_ not in sp_.columns:
        sp_[i_] = ""
K["s"], K["g"], K["e"] = sp_[0].map(norm_sido), sp_[1].map(norm_sgg), sp_[2].map(norm)
K["g2"], K["e2"] = K["g"].map(strip_gu), K["e"].map(strip_num)

# --- 우리 쪽 조회 테이블 ---
tk = dong["행정동명"].str.split(" ")
AD = pd.DataFrame({"adm": dong.index.values, "pop": dong["인구"].values, "s": tk.str[0].map(norm_sido).values,
                   "g": tk.str[1:-1].str.join("").map(norm_sgg).values, "e": tk.str[-1].map(norm).values})
AD["gu"] = dong["시군구명"].str.split(" ", n=1, expand=True)[1].fillna("").map(norm_sgg).values
AD["g2"], AD["e2"] = AD["g"].map(strip_gu), AD["e"].map(strip_num)
ADM_SET = set(AD["adm"])

HAVE_BRIDGE = bridge is not None and all(c in bridge.columns for c in ["법정동코드", "행정동코드", "법정동명", "시군구명", "인구"])
REP, W, S10, S8 = {}, None, set(), set()
if HAVE_BRIDGE:
    bridge = bridge[bridge["법정동코드"].notna()].copy()
    bridge["인구"] = pd.to_numeric(bridge["인구"], errors="coerce")
    P10 = bridge[["법정동코드", "행정동코드", "인구"]].drop_duplicates(["법정동코드", "행정동코드"]).rename(columns={"법정동코드": "key"})
    P8 = P10.assign(key=P10["key"].str[:8]).drop_duplicates(["key", "행정동코드"])
    PP = pd.concat([P10, P8], ignore_index=True)
    PP["인구"] = PP["인구"].fillna(1.0)
    S10, S8 = set(P10["key"]), set(P8["key"])
    REP = dict(zip(*[PP.sort_values(["key", "인구", "행정동코드"], ascending=[True, False, True]).drop_duplicates("key")[c] for c in ["key", "행정동코드"]]))
    W = PP.copy()
    W["w"] = W["인구"] / W.groupby("key")["인구"].transform("sum")
    base = xw[["시도명", "시군구명", "법정동명", "법정동코드"]] if (xw is not None and all(c in xw.columns for c in ["시도명", "시군구명", "법정동명", "법정동코드"])) else None
    if base is None:
        t_ = bridge["시군구명"].str.split(" ", n=1, expand=True)
        base = pd.DataFrame({"시도명": t_[0], "시군구명": t_[1].fillna(""), "법정동명": bridge["법정동명"], "법정동코드": bridge["법정동코드"]})
    XB = base.drop_duplicates().copy()
    XB["s"], XB["g"], XB["e"] = XB["시도명"].map(norm_sido), XB["시군구명"].map(norm_sgg), XB["법정동명"].map(norm)
    al = XB[XB["s"].isin(["전라남도", "광주광역시"])].copy()
    al["s"] = "전남광주통합특별시"
    XB = pd.concat([XB, al], ignore_index=True)
    XB["g2"] = XB["g"].map(strip_gu)
    print("법정동 연결 데이터: 법정동 %d개 → 행정동 (여러 행정동에 걸친 법정동 %d개, 규칙 '%s')" % (len(S10), int((P10.groupby("key")["행정동코드"].nunique() > 1).sum()), MULTI_RULE))
else:
    print("※ 법정동 연결 파일이 없어 '행정동 이름'으로만 연결합니다 (읍·면은 대부분 연결, 도시 동은 근사)")

# --- 1단계: 코드가 있는 경우 ---
def code_variants(c):
    yield c
    p = c[:2]
    if p in ("42", "45"):
        yield {"42": "51", "45": "52"}[p] + c[2:]
    if p in ("51", "52"):
        yield {"51": "42", "52": "45"}[p] + c[2:]
    if c[:5] == "47720":
        yield "27720" + c[5:]
    if c[:5] == "27720":
        yield "47720" + c[5:]

def link_code(c):
    if not c or not c.isdigit():
        return None
    for v in code_variants(c):
        L = len(v)
        if L >= 10:
            if v[:10] in S10:
                return ("bjd", v[:10], "코드: 법정동 10자리")
            if v[:10] in ADM_SET:
                return ("adm", v[:10], "코드: 행정동 10자리")
            if v[:8] in S8:
                return ("bjd", v[:8], "코드: 법정동 8자리(읍면동 단위)")
        elif L == 8:
            if v in S8:
                return ("bjd", v, "코드: 법정동 8자리")
            if v + "00" in ADM_SET:
                return ("adm", v + "00", "코드: 행정동 8자리")
    return None

if (K["code"] != "").any():
    for i_, c_ in K.loc[K["code"] != "", ["code"]].itertuples():
        r_ = link_code(c_)
        if r_:
            K.at[i_, "bjd" if r_[0] == "bjd" else "adm"] = r_[1]
            K.at[i_, "how"] = r_[2]

# --- 2단계: 이름 (법정동명 → 행정동명 → 번호 동 근사) ---
def run_tier(label, kcols, table, tcols, val, target, pick="unique", pool=None):
    t = table if pool is None else pool
    if pick == "unique":
        gg = t.groupby(tcols)[val].agg(["nunique", "first"]).reset_index()
        uq = gg[gg["nunique"] == 1].rename(columns={"first": "_v"})[tcols + ["_v"]]
    else:
        uq = t.sort_values("pop", ascending=False).drop_duplicates(tcols).rename(columns={val: "_v"})[tcols + ["_v"]]
    todo = K["bjd"].isna() & K["adm"].isna() & (K["e"] != "")
    if not todo.any():
        return
    left = K.loc[todo, ["uid"] + kcols].copy()
    left.columns = ["uid"] + tcols
    hit = left.merge(uq, on=tcols, how="inner")
    if len(hit) == 0:
        return
    mp = dict(zip(hit["uid"], hit["_v"]))
    idx = todo & K["uid"].isin(mp)
    K.loc[idx, target] = K.loc[idx, "uid"].map(mp)
    K.loc[idx, "how"] = label

if HAVE_BRIDGE:
    run_tier("법정동명: 시도+시군구+읍면동", ["s", "g", "e"], XB, ["s", "g", "e"], "법정동코드", "bjd")
    run_tier("법정동명: 시군구+읍면동(시도 무시)", ["g", "e"], XB, ["g", "e"], "법정동코드", "bjd")
    run_tier("법정동명: 시군구(일반구 제외)+읍면동", ["g2", "e"], XB, ["g2", "e"], "법정동코드", "bjd")
    run_tier("법정동명: 시도+읍면동(시군구 무시)", ["s", "e"], XB, ["s", "e"], "법정동코드", "bjd")
run_tier("행정동명: 시도+시군구+읍면동", ["s", "g", "e"], AD, ["s", "g", "e"], "adm", "adm")
run_tier("행정동명: 시도+시군구(구 포함)+읍면동", ["s", "g", "e"], AD, ["s", "gu", "e"], "adm", "adm")
run_tier("행정동명: 시군구(일반구 제외)+읍면동", ["g2", "e"], AD, ["g2", "e"], "adm", "adm")
run_tier("행정동명: 시도+읍면동(시군구 무시)", ["s", "e"], AD, ["s", "e"], "adm", "adm")
NUMBERED = AD[AD["e"] != AD["e2"]]
run_tier("번호 동 대표 행정동(근사)", ["s", "g2", "e"], AD, ["s", "g2", "e2"], "adm", "adm", pick="rep", pool=NUMBERED)
run_tier("번호 동 대표 행정동(근사, 시도 무시)", ["g2", "e"], AD, ["g2", "e2"], "adm", "adm", pick="rep", pool=NUMBERED)

LK = K.drop(columns=["uid"])
LK["연결"] = LK["bjd"].notna() | LK["adm"].notna()
tot_kwh = LK["kwh"].sum()
rate = LK.loc[LK["연결"], "kwh"].sum() / tot_kwh if tot_kwh else 0.0
print("=" * 78)
print("지역 연결 결과: 고유 지역 %d개 중 %d개 연결 | kWh 가중 연결률 %.1f%%" % (len(LK), LK["연결"].sum(), 100 * rate))
sm = LK.groupby(LK["how"].replace("", "※ 미연결"))["kwh"].agg(["size", "sum"])
sm["kWh비중"] = (sm["sum"] / tot_kwh * 100).round(1)
print(sm.rename(columns={"size": "지역수", "sum": "kWh"}).to_string())
un = LK[~LK["연결"]].sort_values("kwh", ascending=False)
if len(un):
    print("\n미연결 상위 20 (시도 | 시군구 | 읍면동 | 코드 | kWh):")
    print(un.head(20)[["s", "g", "e", "code", "kwh"]].to_string(index=False))
if rate < 0.9:
    print("\n※ 연결률이 90%% 미만입니다. 위 '미연결 상위' 의 표기(예: 시도 약칭, 읍면동에 '리' 포함, 번호 표기)를 확인하고 알려주세요.")
_lk = LK.drop(columns=["g2", "e2"]).copy()
_lk["name"] = _lk["name"].str.replace("\t", " | ")
_lk = _lk.rename(columns={"code": "한전코드_원본", "name": "한전지역_원본(시도|시군구|읍면동)", "kwh": "kWh합계", "bjd": "연결_법정동코드", "adm": "연결_행정동코드", "how": "연결방법",
                          "s": "시도_정규화", "g": "시군구_정규화", "e": "읍면동_정규화"})
_lk.sort_values("kWh합계", ascending=False).to_csv(os.path.join(OUT_DIR, "내부용", "00_지역연결표.csv"), index=False, encoding="utf-8-sig")
LOG["지역연결률_kWh가중"] = round(float(rate), 4)
LOG["지역연결_방법별"] = {str(k): [int(v["size"]), round(float(v["sum"] / tot_kwh), 4) if tot_kwh else 0] for k, v in sm.iterrows()}

def to_dong(Mx):
    X = Mx.merge(LK[["code", "name", "bjd", "adm"]], on=["code", "name"], how="left")
    a = X[X["adm"].notna()].copy()
    a["행정동코드"], a["w"] = a["adm"], 1.0
    b = X[X["adm"].isna() & X["bjd"].notna()].copy()
    if len(b):
        if MULTI_RULE == "split" and W is not None:
            b = b.merge(W[["key", "행정동코드", "w"]], left_on="bjd", right_on="key", how="inner")
        else:
            b["행정동코드"], b["w"] = b["bjd"].map(REP), 1.0
            b = b[b["행정동코드"].notna()]
    out = pd.concat([a, b], ignore_index=True)
    for c in ["kwh", "night", "cust"]:
        out[c] = out[c] * out["w"]
    return out.groupby(["행정동코드", "month", "kind"], as_index=False).agg(kwh=("kwh", "sum"), night=("night", "sum"), cust=("cust", "sum"))

In [ ]:
# [7] 행정동 × 월 재집계 + 품질 점검
E = to_dong(M)
def kind_key(k):
    return "nonbiz" if ("비" in k and "사업" in k) else ("biz" if "사업" in k else "tot")
E["kk"] = E["kind"].map(kind_key)
pv = E.pivot_table(index=["행정동코드", "month"], columns="kk", values=["kwh", "night", "cust"], aggfunc="sum")
pv.columns = ["%s_%s" % (a, b) for a, b in pv.columns]
T = pv.reset_index()
for c in ["kwh", "night", "cust"]:
    for k in ["nonbiz", "biz", "tot"]:
        if "%s_%s" % (c, k) not in T.columns:
            T["%s_%s" % (c, k)] = 0.0
T = T.fillna(0.0)
T["kwh"] = T["kwh_nonbiz"] + T["kwh_biz"] + T["kwh_tot"]
T["night"] = T["night_nonbiz"] + T["night_biz"] + T["night_tot"]
grid = pd.MultiIndex.from_product([dong.index, DATA_MONTHS], names=["행정동코드", "month"])
T = T.set_index(["행정동코드", "month"]).reindex(grid, fill_value=0.0).reset_index()
HAS = set(E["행정동코드"])
dong["한전자료있음"] = dong.index.isin(HAS)
POP = dong["인구"].values.astype(float)

def wide(col):
    return T.pivot(index="행정동코드", columns="month", values=col).reindex(index=dong.index, columns=DATA_MONTHS).fillna(0.0)
KW, KWN, KWB = wide("kwh"), wide("kwh_nonbiz"), wide("kwh_biz")
NIGHTW = wide("night")
SA = (KW.values / POP[:, None]) * 1000          # 인구 1천 명당 월 kWh (행정동 × 월)
S = pd.DataFrame(SA, index=dong.index, columns=DATA_MONTHS)
W12 = list(range(max(0, len(DATA_MONTHS) - 12), len(DATA_MONTHS)))     # 최근 12개월 위치
p12 = SA[:, W12].mean(axis=1)                                          # 최근 12개월 월평균(1천 명당)
kw12 = KW.values[:, W12].sum(axis=1)
# 한전 월 위치 → 주유소 월별 거리 행렬의 열 위치 (범위 밖은 양 끝으로 보정)
GI = np.array([0 if m_ < GAS_MONTHS[0] else (len(GAS_MONTHS) - 1 if m_ > GAS_MONTHS[-1] else int(GAS_MONTHS.get_loc(m_))) for m_ in DATA_MONTHS])

print("=" * 78)
print("[품질 점검]")
q_ = []
def chk(name, val, note=""):
    q_.append(dict(항목=name, 값=val, 비고=note))
    print("  %-34s %s %s" % (name, val, note))
chk("한전 지역 → 행정동 연결률(kWh 가중)", "%.1f%%" % (100 * LOG["지역연결률_kWh가중"]), "90% 미만이면 미연결 목록 확인")
chk("한전 자료가 붙은 행정동", "%d / %d" % (len(HAS), len(dong)))
chk("총 kWh (입력 월집계)", format(M["kwh"].sum(), ",.0f"))
chk("총 kWh (행정동 반영)", format(KW.values.sum(), ",.0f"), "차이 = 미연결분")
chk("kWh 0 인 행정동-월 비율", "%.1f%%" % (100 * (KW.values == 0).mean()))
tot_stats = {k: sum(v[k] for v in STATS.values()) for k in ["rows", "masked", "cells", "neg", "zero", "bad_month", "dropped_rows"]}
chk("숫자가 아닌 값(마스킹 추정) 비율", "%.3f%%" % (100 * tot_stats["masked"] / max(tot_stats["cells"], 1)), "높으면 소규모 셀 비공개 처리 가능성")
chk("음수 kWh 칸 수", tot_stats["neg"])
chk("날짜 해석 실패 행", tot_stats["bad_month"])
for km_key, lab in [("nonbiz", "비사업자"), ("biz", "사업자")]:
    if (T["kwh_" + km_key] != 0).any():
        chk("%s kWh 비중" % lab, "%.1f%%" % (100 * T["kwh_" + km_key].sum() / max(T["kwh"].sum(), 1e-9)))
chk("고객호수 정보", "있음" if T[["cust_nonbiz", "cust_biz", "cust_tot"]].values.sum() > 0 else "없음")
gstrict = ((D_ > 5) & (N_ == 0))[:, -1]
chk("엄격 공백(5km) 행정동 중 한전 자료 있음", "%d / %d" % (int(dong["한전자료있음"].values[gstrict].sum()), int(gstrict.sum())), "없는 곳은 충전 0 으로 간주")
yr = T.assign(y=T["month"].dt.year).groupby("y")["kwh"].sum()
chk("연도별 kWh", {int(k): round(float(v)) for k, v in yr.items()})
QC = pd.DataFrame(q_)
QC.to_csv(os.path.join(OUT_DIR, "반출후보", "01_품질점검.csv"), index=False, encoding="utf-8-sig")
_t = T.assign(kwh_per1000=(T["kwh"].values / np.repeat(POP, len(DATA_MONTHS)) * 1000))
_t["month"] = _t["month"].dt.strftime("%Y-%m")
_t = _t.rename(columns={"month": "월", "kwh": "합계_kWh", "night": "합계_야간kWh", "kwh_nonbiz": "비사업자_kWh", "kwh_biz": "사업자_kWh", "kwh_tot": "구분없음_kWh",
                        "night_nonbiz": "비사업자_야간kWh", "night_biz": "사업자_야간kWh", "night_tot": "구분없음_야간kWh", "cust_nonbiz": "비사업자_고객호수", "cust_biz": "사업자_고객호수",
                        "cust_tot": "구분없음_고객호수", "kwh_per1000": "인구1천명당_kWh"})
_t.to_csv(os.path.join(OUT_DIR, "내부용", "02_행정동_월별_충전량.csv"), index=False, encoding="utf-8-sig")
LOG["품질"] = {r["항목"]: str(r["값"]) for r in q_}

In [ ]:
# [8] 지표 01 — 충전기 '개수' vs 실제 'kWh' 불평등 (지니계수·로렌츠)
def lorenz(x, w=None):
    x = np.asarray(x, float)
    w = np.ones_like(x) if w is None else np.asarray(w, float)
    m_ = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m_], w[m_]
    if x.sum() <= 0:
        return np.array([0, 1.0]), np.array([0, 1.0])
    o = np.argsort(x, kind="mergesort")
    x, w = x[o], w[o]
    return np.r_[0, np.cumsum(w) / w.sum()], np.r_[0, np.cumsum(x * w) / (x * w).sum()]

def gini(x, w=None):
    cw, cx = lorenz(x, w)
    return float(1 - np.sum((cw[1:] - cw[:-1]) * (cx[1:] + cx[:-1])))

cnt = dong["충전기개소수_현재"].fillna(0).values
items = [("충전기 개소수(현재)", cnt), ("kWh 최근12개월(전체)", kw12), ("kWh 최근12개월(비사업자=자가충전)", KWN.values[:, W12].sum(axis=1)),
         ("kWh 최근12개월(사업자=공용)", KWB.values[:, W12].sum(axis=1)), ("인구 1천명당 kWh(최근12개월)", p12), ("인구 1천명당 충전기 개소수", cnt / POP * 1000)]
rows = []
for lab, x in items:
    rows.append(dict(지표=lab, 지니_행정동동일가중=round(gini(x), 4), 지니_인구가중=round(gini(x, POP), 4)))
GINI = pd.DataFrame(rows)
print(GINI.to_string(index=False))
yrs = sorted(set(m_.year for m_ in DATA_MONTHS))
gy = []
for y in yrs:
    cols_ = [i for i, m_ in enumerate(DATA_MONTHS) if m_.year == y]
    if len(cols_) >= 6:
        v = KW.values[:, cols_].sum(axis=1)
        gy.append(dict(연도=y, 개월수=len(cols_), 지니_kWh=round(gini(v), 4), 지니_인구가중=round(gini(v, POP), 4), 지니_인구당kWh_인구가중=round(gini(v / POP, POP), 4)))
GINY = pd.DataFrame(gy)
print(GINY.to_string(index=False))
r_ = pd.Series(cnt).rank().corr(pd.Series(kw12).rank())
print("충전기 개소수와 kWh 의 순위상관(스피어만): %.3f  (낮을수록 '설치는 됐지만 안 쓰이는' 곳이 많다는 신호)" % r_)
x1, y1 = lorenz(cnt)
x2, y2 = lorenz(kw12)
n_ = max(len(x1), len(x2))
LZ = pd.DataFrame({"누적행정동비율_개소수": np.interp(np.linspace(0, 1, 101), x1, x1), "누적비율_개소수": np.interp(np.linspace(0, 1, 101), x1, y1),
                   "누적행정동비율_kWh": np.interp(np.linspace(0, 1, 101), x2, x2), "누적비율_kWh": np.interp(np.linspace(0, 1, 101), x2, y2)})
GINI.to_csv(os.path.join(OUT_DIR, "반출후보", "03_지니계수.csv"), index=False, encoding="utf-8-sig")
GINY.to_csv(os.path.join(OUT_DIR, "반출후보", "03_지니계수_연도별.csv"), index=False, encoding="utf-8-sig")
LZ.to_csv(os.path.join(OUT_DIR, "반출후보", "03_로렌츠곡선_점.csv"), index=False, encoding="utf-8-sig")
LOG["지니"] = GINI.set_index("지표")["지니_행정동동일가중"].to_dict()
LOG["개소수_kWh_순위상관"] = round(float(r_), 3)

In [ ]:
# [9] 지표 04 — 가동률과 '유령 충전기' 탐지 (설치는 됐는데 거의 안 쓰이는 곳)
# 가동률 = 최근 12개월 월평균 kWh ÷ 충전기 개소수  (개소수는 환경공단 설치현황 기준, 2시점 스냅샷이라 근사치)
mo_kwh = kw12 / max(len(W12), 1)
rate = np.where(cnt > 0, mo_kwh / np.where(cnt > 0, cnt, 1), np.nan)
zero_m = (KW.values[:, W12] == 0).sum(axis=1)
ok_ = cnt > 0
GH = {}
for q in [0.05, 0.10, 0.20]:
    GH[q] = np.nanquantile(rate[ok_ & dong["한전자료있음"].values], q) if (ok_ & dong["한전자료있음"].values).any() else 0.0
thr_ghost = GH[GHOST_Q] if GHOST_Q in GH else float(np.nanquantile(rate[ok_ & dong["한전자료있음"].values], GHOST_Q))
has_ = dong["한전자료있음"].values
ghost = ok_ & has_ & (rate <= thr_ghost)
long_zero = ok_ & has_ & (zero_m >= len(W12) - 1)
no_data_with_chargers = ok_ & ~has_
GHT = dong[["행정동명", "시군구명", "인구", "충전기개소수_현재"]].copy()
GHT["월평균kWh"] = mo_kwh.round(2)
GHT["가동률"] = np.round(rate, 3)
GHT["최근12개월_무사용월수"] = zero_m
GHT["유령충전기_후보"] = ghost
GHT["장기_무사용"] = long_zero
GHT["한전자료있음"] = has_
print("가동률 하위 %d%% 기준 = %.2f kWh/월/개소 → 유령 충전기 후보 행정동 %d곳(인구 %s명), 최근 12개월 내내 kWh 0 인 곳 %d곳" % (
    GHOST_Q * 100, thr_ghost, ghost.sum(), format(int(POP[ghost].sum()), ","), long_zero.sum()))
print("※ 충전기는 있는데 한전 자료 행이 아예 없는 행정동 %d곳(연결 누락 또는 사용 0)은 후보에서 별도 취급합니다" % no_data_with_chargers.sum())
print(pd.DataFrame({"기준분위": list(GH), "가동률기준(kWh/월/개소)": [round(v, 3) for v in GH.values()],
                    "후보 행정동수": [int((ok_ & dong["한전자료있음"].values & (rate <= v)).sum()) for v in GH.values()]}).to_string(index=False))
GHT.reset_index().to_csv(os.path.join(OUT_DIR, "내부용", "04_가동률_유령충전기.csv"), index=False, encoding="utf-8-sig")
LOG["유령충전기"] = dict(기준=round(float(thr_ghost), 3), 후보_행정동=int(ghost.sum()), 후보_인구=int(POP[ghost].sum()), 장기무사용_행정동=int(long_zero.sum()), 충전기있고_한전자료없음=int(no_data_with_chargers.sum()))

In [ ]:
# [10] 주유소 접근성 공백 분류(엄격/중심점 두 정의) + 충전 실사용 기준 + 4분면
def classify_ab(above):
    lt = above[:, 0]
    suf = np.flip(np.cumprod(np.flip(above, 1), axis=1), 1).astype(bool)
    has = suf.any(axis=1)
    first = np.where(has, suf.argmax(axis=1), -1)
    status = np.where(lt, "left_truncated", np.where(has, "event", "censored"))
    return status, first

DEFS = {"strict": "엄격(경계 안 주유소 0개 + 중심점 거리 초과, 하한)", "centroid": "중심점 거리 초과(상한)"}
GAP = {}
for km in sorted(set(GAP_KMS + [PRIMARY_GAP_KM])):
    for dfn in DEFS:
        ab = ((D_ > km) & (N_ == 0)) if dfn == "strict" else (D_ > km)
        st_, f_ = classify_ab(ab)
        GAP[(dfn, km)] = dict(status=st_, first=f_, above=ab)
LATEST = int(GI[-1])
_pos = p12[(POP > 0) & (p12 > 0)]
if len(_pos) < 20:
    _pos = p12[POP > 0]
THR12 = {q: float(np.nanquantile(_pos, q)) for q in sorted(set(CHARGE_QS + [PRIMARY_CHARGE_Q]))}
print("충전 실사용 기준(인구 1천명당 월 kWh) = 최근 12개월 사용량이 있는 행정동(%d곳) 분포의 분위수:" % len(_pos), {q: round(v, 3) for q, v in THR12.items()})
print("   (사용량 0 인 행정동 %d곳은 항상 '충전 낮음'으로 봅니다)" % int(((POP > 0) & (p12 <= 0)).sum()))

def mask_df(df, cols, k=None):
    k = MIN_CELL if k is None else k
    out = df.copy()
    for c in cols:
        out[c] = out[c].astype(object)
        out.loc[pd.to_numeric(out[c], errors="coerce") < k, c] = "<%d" % k
    return out

gnow = {d: GAP[(d, PRIMARY_GAP_KM)]["above"][:, LATEST] for d in DEFS}
low = p12 <= THR12[PRIMARY_CHARGE_Q]
QD = dong[["행정동명", "시도", "시군구명", "인구", "65세이상", "고령비율", "인구감소지역"]].copy()
QD["최근12개월_1천명당월kWh"] = np.round(p12, 3)
QD["충전_낮음"] = low
for d in DEFS:
    QD["공백_현재_" + d] = gnow[d]
    QD["공백상태_" + d] = GAP[(d, PRIMARY_GAP_KM)]["status"]
    QD["4분면_" + d] = np.select([~gnow[d] & ~low, ~gnow[d] & low, gnow[d] & ~low, gnow[d] & low],
                                 ["둘 다 문제없음", "아직 기름만 씀(미전환)", "전기로 잘 넘어감(전환 성공)", "이중공백"], default="")
QD["한전자료있음"] = dong["한전자료있음"].values
rows = []
for d in DEFS:
    for lab, g_ in QD.groupby("4분면_" + d):
        rows.append(dict(공백정의=d, 분류=lab, 행정동수=len(g_), 인구=int(g_["인구"].sum()), 인구65이상=int(g_["65세이상"].sum())))
QS = pd.DataFrame(rows)
print("[4분면] 공백 기준 %dkm, 충전 낮음 = 최근12개월 1천명당 kWh ≤ 사용 행정동 %d분위(%.3f)" % (PRIMARY_GAP_KM, PRIMARY_CHARGE_Q * 100, THR12[PRIMARY_CHARGE_Q]))
print(QS.pivot_table(index="분류", columns="공백정의", values=["행정동수", "인구"], aggfunc="sum").fillna(0).astype(int).to_string())
QD.reset_index().to_csv(os.path.join(OUT_DIR, "내부용", "05_행정동_공백_충전_분류.csv"), index=False, encoding="utf-8-sig")
mask_df(QS, ["행정동수"]).to_csv(os.path.join(OUT_DIR, "반출후보", "05_4분면_요약.csv"), index=False, encoding="utf-8-sig")
LOG["4분면"] = {d: {r["분류"]: [int(r["행정동수"]), int(r["인구"])] for _, r in QS[QS["공백정의"] == d].iterrows()} for d in DEFS}

In [ ]:
# [11] 지표 02 — 전환 시차(생존분석): 공백 시작 → 충전 실사용 시작
def first_run_idx(A, thr, run):
    ok = (A >= thr).astype(int)
    c = np.cumsum(np.c_[np.zeros((A.shape[0], 1)), ok], axis=1)
    if A.shape[1] < run:
        return -np.ones(A.shape[0], dtype=int)
    hit = (c[:, run:] - c[:, :-run]) >= run
    return np.where(hit.any(axis=1), hit.argmax(axis=1), -1)

def km_table(dur, ev, entry=None):
    dur = np.asarray(dur, float)
    ev = np.asarray(ev, int)
    entry = np.zeros_like(dur) if entry is None else np.asarray(entry, float)
    S_, var, out = 1.0, 0.0, []
    for t in np.unique(dur[ev == 1]):
        n = np.sum((entry < t) & (dur >= t))
        d = np.sum((dur == t) & (ev == 1))
        if n == 0:
            continue
        S_ *= (1 - d / n)
        if n > d:
            var += d / (n * (n - d))
        se = S_ * np.sqrt(var)
        out.append((t, int(n), int(d), S_, max(0.0, S_ - 1.96 * se), min(1.0, S_ + 1.96 * se)))
    return pd.DataFrame(out, columns=["개월", "위험집합", "사건수", "생존확률", "하한95", "상한95"])

def km_median(km):
    b = km[km["생존확률"] <= 0.5]
    return float(b["개월"].iloc[0]) if len(b) else np.nan

def logrank(t1, e1, t2, e2):
    import math
    t1, e1, t2, e2 = [np.asarray(v) for v in (t1, e1, t2, e2)]
    times = np.unique(np.r_[t1[e1 == 1], t2[e2 == 1]])
    O = Ex = V = 0.0
    for t in times:
        n1, n2 = (t1 >= t).sum(), (t2 >= t).sum()
        n = n1 + n2
        d1, d2 = ((t1 == t) & (e1 == 1)).sum(), ((t2 == t) & (e2 == 1)).sum()
        d = d1 + d2
        if n < 2:
            continue
        O += d1
        Ex += d * n1 / n
        V += d * (n1 / n) * (1 - n1 / n) * (n - d) / (n - 1)
    if V <= 0:
        return np.nan, np.nan
    chi = (O - Ex) ** 2 / V
    return float(chi), float(math.erfc(math.sqrt(chi / 2)))

def onset_table(dfn, km, q):
    """공백 시작이 관측(event)되었거나 관측 이전부터(left_truncated) 공백인 행정동의 시차 표"""
    g = GAP[(dfn, km)]
    cs = first_run_idx(SA, THR12[q], RUN_MONTHS)
    rows = []
    for i in np.where(g["status"] != "censored")[0]:
        stt, cidx = g["status"][i], cs[i]
        rec = dict(행정동코드=dong.index[i], 행정동명=dong["행정동명"].iloc[i], 공백정의=dfn, 기준km=km, 충전기준분위=q, 상태=stt, 공백시작=None, 충전시작=None,
                   진입=0, 기간_개월=np.nan, 사건=np.nan, 비고="")
        cstart = DATA_MONTHS[cidx] if cidx >= 0 else None
        rec["충전시작"] = None if cstart is None else cstart.strftime("%Y-%m")
        if stt == "event":
            gstart = GAS_MONTHS[g["first"][i]]
            rec["공백시작"] = gstart.strftime("%Y-%m")
            rec["진입"] = max(0, months_between(gstart, DATA_START))
            if gstart > DATA_END:
                rec["비고"] = "공백 시작이 한전 자료 기간 이후"
            elif cstart is None:
                rec.update(기간_개월=months_between(gstart, DATA_END), 사건=0, 비고="관측 끝까지 충전 시작 없음(중도절단)")
            elif cidx == 0 and rec["진입"] > 0:
                rec["비고"] = "한전 자료 시작 시점에 이미 충전 중 → 시작 시점 불명(분석 제외)"
            elif cstart < gstart:
                rec.update(기간_개월=0, 사건=1, 비고="공백 시작 전에 이미 충전 시작")
            else:
                rec.update(기간_개월=months_between(gstart, cstart), 사건=1)
        else:
            rec["공백시작"] = "%s 이전(불명)" % GAS_MONTHS[0].strftime("%Y-%m")
            if cstart is None:
                rec.update(기간_개월=months_between(DATA_START, DATA_END), 사건=0, 비고="관측 이전부터 공백 / 충전 시작 없음(하한값)")
            else:
                rec.update(기간_개월=months_between(DATA_START, cstart), 사건=1, 비고="관측 이전부터 공백(하한값: 한전 자료 시작부터 경과)")
        rows.append(rec)
    return pd.DataFrame(rows)

def summarize_onset(df):
    out = []
    for stt in ["event", "left_truncated"]:
        s = df[(df["상태"] == stt) & df["기간_개월"].notna()]
        km = km_table(s["기간_개월"], s["사건"], s["진입"]) if len(s) else pd.DataFrame()
        out.append(dict(상태=stt, 대상=int((df["상태"] == stt).sum()), 분석포함=len(s), 충전시작_관측=int(s["사건"].sum()) if len(s) else 0,
                        중앙값_개월=km_median(km) if len(km) else np.nan))
    return pd.DataFrame(out)

ON = {d: onset_table(d, PRIMARY_GAP_KM, PRIMARY_CHARGE_Q) for d in DEFS}
ON_ALL = pd.concat(ON.values(), ignore_index=True)
print("[공백 시작 이후 충전 시작까지의 시차] 공백 %dkm / 충전 기준 %d분위 / 연속 %d개월" % (PRIMARY_GAP_KM, PRIMARY_CHARGE_Q * 100, RUN_MONTHS))
SUMM = []
for d in DEFS:
    sm_ = summarize_onset(ON[d])
    sm_.insert(0, "공백정의", d)
    SUMM.append(sm_)
    print(sm_.to_string(index=False))
SUMM = pd.concat(SUMM, ignore_index=True)
KMS = []
for d in DEFS:
    for stt in ["event", "left_truncated"]:
        s = ON[d][(ON[d]["상태"] == stt) & ON[d]["기간_개월"].notna()]
        if len(s) and s["사건"].sum() > 0:
            k_ = km_table(s["기간_개월"], s["사건"], s["진입"])
            k_.insert(0, "상태", stt)
            k_.insert(0, "공백정의", d)
            KMS.append(k_)
ON_ALL.to_csv(os.path.join(OUT_DIR, "내부용", "06_전환시차_개별_행정동.csv"), index=False, encoding="utf-8-sig")
mask_df(SUMM, ["대상", "분석포함", "충전시작_관측"]).to_csv(os.path.join(OUT_DIR, "반출후보", "06_전환시차_요약.csv"), index=False, encoding="utf-8-sig")
(pd.concat(KMS, ignore_index=True) if KMS else pd.DataFrame()).to_csv(os.path.join(OUT_DIR, "반출후보", "06_전환시차_KM곡선.csv"), index=False, encoding="utf-8-sig")

# --- 달력 시간 생존분석(전체 행정동): 한전 자료 시작 후 충전 실사용에 도달하기까지, 공백 지역 vs 그 외 ---
cs_all = first_run_idx(SA, THR12[PRIMARY_CHARGE_Q], RUN_MONTHS)
valid = (POP > 0) & (cs_all != 0)                  # 자료 시작 시점에 이미 충전 중인 곳은 제외(시작 시점 불명)
cal_dur = np.where(cs_all >= 0, cs_all, len(DATA_MONTHS) - 1).astype(float)
cal_ev = (cs_all >= 0).astype(int)
CAL, COX = [], []
print("\n[달력 시간 생존분석] 한전 자료 시작 후 '충전 실사용' 도달까지 (대상 %d개 행정동)" % valid.sum())
for d in DEFS:
    gmask = gnow[d] & valid
    rmask = (~gnow["centroid"]) & valid          # 대조군 = 중심점 기준으로도 공백이 아닌 행정동
    gk, rk = km_table(cal_dur[gmask], cal_ev[gmask]), km_table(cal_dur[rmask], cal_ev[rmask])
    chi, p = logrank(cal_dur[gmask], cal_ev[gmask], cal_dur[rmask], cal_ev[rmask])
    print("  %-9s 공백 %d곳(충전 도달 %d) vs 그 외 %d곳(충전 도달 %d) | 중앙값 %s vs %s개월 | log-rank p=%s" % (
        d, gmask.sum(), cal_ev[gmask].sum(), rmask.sum(), cal_ev[rmask].sum(), km_median(gk) if len(gk) else "NA", km_median(rk) if len(rk) else "NA", "%.4f" % p if p == p else "NA"))
    for lab, k_ in [("공백(%s)" % d, gk), ("공백 아님", rk)]:
        if len(k_):
            CAL.append(k_.assign(집단=lab))
    COX.append(dict(공백정의=d, 공백_행정동=int(gmask.sum()), 공백_충전도달=int(cal_ev[gmask].sum()), 대조_행정동=int(rmask.sum()), 대조_충전도달=int(cal_ev[rmask].sum()),
                    공백_중앙값개월=km_median(gk) if len(gk) else np.nan, 대조_중앙값개월=km_median(rk) if len(rk) else np.nan, logrank_p=p))
COXD = pd.DataFrame(COX)
if OPT.get("statsmodels"):
    try:
        from statsmodels.duration.hazard_regression import PHReg
        X = pd.DataFrame({"공백(엄격)": gnow["strict"].astype(float), "고령비율": dong["고령비율"].values, "ln인구": np.log(POP.clip(min=1)),
                          "인구감소지역": (dong["인구감소지역"] == "Y").astype(float).values, "ln(1+충전기수)": np.log1p(cnt)})
        m_ = valid & X.notna().all(axis=1).values
        res_ = PHReg(cal_dur[m_] + 0.5, X[m_].values, status=cal_ev[m_], ties="breslow").fit(disp=False)
        ci = res_.conf_int()
        cx = pd.DataFrame({"변수": X.columns, "위험비(HR)": np.exp(res_.params), "하한95": np.exp(ci[:, 0]), "상한95": np.exp(ci[:, 1]), "p값": res_.pvalues})
        print("\n[Cox 비례위험] 위험비 < 1 이면 충전 실사용 도달이 늦다는 뜻\n", cx.round(4).to_string(index=False))
        cx.to_csv(os.path.join(OUT_DIR, "반출후보", "06_Cox_결과.csv"), index=False, encoding="utf-8-sig")
        LOG["Cox"] = cx.round(4).to_dict("records")
    except Exception as e_:
        print("Cox 회귀 생략:", e_)
else:
    print("(statsmodels 가 없어 Cox 회귀는 생략 — Kaplan-Meier·log-rank 결과를 사용하세요)")
mask_df(COXD, ["공백_행정동", "공백_충전도달"]).to_csv(os.path.join(OUT_DIR, "반출후보", "06_달력시간_생존_요약.csv"), index=False, encoding="utf-8-sig")
(pd.concat(CAL, ignore_index=True) if CAL else pd.DataFrame()).to_csv(os.path.join(OUT_DIR, "반출후보", "06_달력시간_KM곡선.csv"), index=False, encoding="utf-8-sig")
LOG["전환시차_요약"] = SUMM.to_dict("records")
LOG["달력시간_생존"] = COXD.round(4).to_dict("records")

In [ ]:
# [12] 지표 05 — 이중공백 노출 인구·개월과 우선순위 (공백 × 충전 낮음이 겹친 달)
SAr = pd.DataFrame(SA).T.rolling(3, min_periods=1).mean().T.values       # 3개월 이동평균(월별 변동 완화)
LOWM = SAr <= THR12[PRIMARY_CHARGE_Q]
DBL = {}
for d in DEFS:
    gm_ = GAP[(d, PRIMARY_GAP_KM)]["above"][:, GI]
    DBL[d] = gm_ & LOWM
EXP = dong[["행정동명", "시도", "시군구코드", "시군구명", "인구", "65세이상", "고령비율"]].copy()
for d in DEFS:
    mths = DBL[d].sum(axis=1)
    EXP["이중공백_개월_" + d] = mths
    EXP["노출_인구개월_" + d] = POP * mths
    EXP["노출_65세이상_인구개월_" + d] = dong["65세이상"].values * mths
    EXP["이중공백_현재_" + d] = DBL[d][:, -1]
    EXP["공백만_개월_" + d] = GAP[(d, PRIMARY_GAP_KM)]["above"][:, GI].sum(axis=1)
EXP.reset_index().to_csv(os.path.join(OUT_DIR, "내부용", "07_이중공백_노출_행정동.csv"), index=False, encoding="utf-8-sig")
agg = {}
for d in DEFS:
    for lvl, key in [("시군구", "시군구명"), ("시도", "시도")]:
        a = EXP.groupby(key).agg(행정동수=("인구", "size"), 이중공백_현재_행정동수=("이중공백_현재_" + d, "sum"), 노출_인구개월=("노출_인구개월_" + d, "sum"),
                                 노출_65세이상_인구개월=("노출_65세이상_인구개월_" + d, "sum")).reset_index().rename(columns={key: "지역"})
        a = a.sort_values("노출_인구개월", ascending=False)
        a.insert(0, "수준", lvl)
        a.insert(0, "공백정의", d)
        agg[(d, lvl)] = a
allagg = pd.concat(agg.values(), ignore_index=True)
mask_df(allagg, ["이중공백_현재_행정동수"]).to_csv(os.path.join(OUT_DIR, "반출후보", "07_이중공백_노출_지역별.csv"), index=False, encoding="utf-8-sig")
for d in DEFS:
    tot_pm = EXP["노출_인구개월_" + d].sum()
    cur_ = EXP["이중공백_현재_" + d]
    print("[이중공백 노출 | %s] 현재 이중공백 행정동 %d곳(인구 %s명, 65세 이상 %s명) | 누적 노출 %s 인구·개월 (평균 %.1f개월)" % (
        d, int(cur_.sum()), format(int(POP[cur_.values].sum()), ","), format(int(dong["65세이상"].values[cur_.values].sum()), ","), format(int(tot_pm), ","),
        tot_pm / max(POP[cur_.values].sum(), 1)))
print("\n노출 인구·개월 상위 10 시군구 (엄격 정의):")
print(agg[("strict", "시군구")].head(10).to_string(index=False))
LOG["이중공백_현재"] = {d: dict(행정동=int(EXP["이중공백_현재_" + d].sum()), 인구=int(POP[EXP["이중공백_현재_" + d].values].sum())) for d in DEFS}

In [ ]:
# [13] 정책 유형 분류 (인프라 부재형 / 유휴 충전기형 / 고령 취약형 / 공백 임박형)  — 엄격 정의 기준, 규칙은 임의이므로 값은 조정 가능
FAR_CHARGER_KM = 3.0                        # 충전기까지 이 거리 이상이면 '충전 인프라 멀다'
nb_share = np.where(kw12 > 0, KWN.values[:, W12].sum(axis=1) / np.where(kw12 > 0, kw12, 1), np.nan)
q75_old = np.nanquantile(dong["고령비율"].values, 0.75)
selfc_low = nb_share <= np.nanmedian(nb_share)          # 자가충전(비사업자) 비중이 중앙값 이하
dbl_now = DBL["strict"][:, -1]
gap_now_s = gnow["strict"]
dist_now = D_[:, LATEST]
dist_3y = D_[:, max(0, LATEST - 36)]
cd = dong["충전기거리_현재"].fillna(0).values
TY = pd.DataFrame(index=dong.index)
TY["① 인프라 부재형"] = dbl_now & ((cnt == 0) | (cd >= FAR_CHARGER_KM))
TY["② 유휴 충전기형"] = dbl_now & (cnt >= 1) & (cd < FAR_CHARGER_KM) & (rate <= thr_ghost)
TY["③ 고령 취약형"] = dbl_now & (dong["고령비율"].values >= q75_old) & selfc_low
TY["④ 공백 임박형"] = (~gap_now_s) & (dist_now >= 3) & (dist_now < PRIMARY_GAP_KM) & (dist_now > dist_3y) & low
TY["전환 성공(공백이나 충전 활발)"] = gap_now_s & ~low
order_ = ["① 인프라 부재형", "② 유휴 충전기형", "③ 고령 취약형", "④ 공백 임박형", "전환 성공(공백이나 충전 활발)"]
TY["주유형"] = "해당없음"
for lab in reversed(order_):
    TY.loc[TY[lab], "주유형"] = lab
TY.insert(0, "행정동명", dong["행정동명"].values)
TY.insert(1, "시군구명", dong["시군구명"].values)
TY["인구"] = POP
TY["65세이상"] = dong["65세이상"].values
TY["현재_주유소거리_km"] = np.round(dist_now, 2)
TY["최근12개월_1천명당월kWh"] = np.round(p12, 3)
TY["충전기개소수"] = cnt
TY["충전기거리_km"] = cd
TY["가동률"] = np.round(rate, 3)
TY["고령비율"] = dong["고령비율"].values
TY["자가충전비중"] = np.round(nb_share, 3)
TY.reset_index().to_csv(os.path.join(OUT_DIR, "내부용", "08_정책유형_행정동.csv"), index=False, encoding="utf-8-sig")
tys = TY[TY["주유형"] != "해당없음"].groupby("주유형").agg(행정동수=("인구", "size"), 인구=("인구", "sum"), 인구65=("65세이상", "sum")).reset_index()
print(tys.to_string(index=False))
mask_df(tys, ["행정동수"]).to_csv(os.path.join(OUT_DIR, "반출후보", "08_정책유형_요약.csv"), index=False, encoding="utf-8-sig")
LOG["정책유형"] = {r["주유형"]: [int(r["행정동수"]), int(r["인구"])] for _, r in tys.iterrows()}

In [ ]:
# [14] 자가충전(비사업자) 비중과 야간충전 — 집에서 충전하는 비중이 지역마다 다른가
NNW, BNW = wide("night_nonbiz"), wide("night_biz")
groups = {"전국": np.ones(len(dong), bool), "인구감소지역": (dong["인구감소지역"] == "Y").values, "엄격 공백(현재)": gnow["strict"],
          "중심점만 공백(현재)": gnow["centroid"] & ~gnow["strict"], "이중공백(현재, 엄격)": dbl_now}
rows = []
for lab, m_ in groups.items():
    if m_.sum() == 0:
        continue
    k_, kn, kb = kw12[m_].sum(), KWN.values[:, W12].sum(axis=1)[m_].sum(), KWB.values[:, W12].sum(axis=1)[m_].sum()
    nn_, nb_ = NNW.values[:, W12].sum(axis=1)[m_].sum(), BNW.values[:, W12].sum(axis=1)[m_].sum()
    rows.append(dict(집단=lab, 행정동수=int(m_.sum()), 인구=int(POP[m_].sum()), 최근12개월_1천명당_월kWh=round(k_ / 12 / POP[m_].sum() * 1000, 3) if POP[m_].sum() else np.nan,
                     비사업자_비중=round(kn / k_, 4) if k_ else np.nan, 야간비중_비사업자=round(nn_ / kn, 4) if kn else np.nan, 야간비중_사업자=round(nb_ / kb, 4) if kb else np.nan))
SELF = pd.DataFrame(rows)
print(SELF.to_string(index=False))
mask_df(SELF, ["행정동수"]).to_csv(os.path.join(OUT_DIR, "반출후보", "09_자가충전_야간충전.csv"), index=False, encoding="utf-8-sig")
LOG["자가충전"] = SELF.to_dict("records")

In [ ]:
# [15] 월별 추세 — 공백 지역과 그 외의 충전 격차가 커지고 있는가
TSG = {"전국": np.ones(len(dong), bool), "인구감소지역": (dong["인구감소지역"] == "Y").values, "인구감소 아닌 곳": (dong["인구감소지역"] != "Y").values,
       "엄격 공백(현재)": gnow["strict"], "이중공백(현재, 엄격)": dbl_now}
rows = []
for j, m_ in enumerate(DATA_MONTHS):
    for lab, msk in TSG.items():
        if msk.sum() == 0:
            continue
        kk = KW.values[msk, j].sum()
        rows.append(dict(월=m_.strftime("%Y-%m"), 집단=lab, 행정동수=int(msk.sum()), 총kWh=round(kk, 1), **{"1천명당_kWh": round(kk / POP[msk].sum() * 1000, 4),
                         "비사업자비중": round(KWN.values[msk, j].sum() / kk, 4) if kk else np.nan}))
TSR = pd.DataFrame(rows)
mask_df(TSR, ["행정동수"]).to_csv(os.path.join(OUT_DIR, "반출후보", "10_월별_추세.csv"), index=False, encoding="utf-8-sig")
last_ = TSR[TSR["월"] == DATA_END.strftime("%Y-%m")].set_index("집단")["1천명당_kWh"]
first_ = TSR[TSR["월"] == DATA_START.strftime("%Y-%m")].set_index("집단")["1천명당_kWh"]
print(pd.DataFrame({"시작월": first_, "마지막월": last_}).round(3).to_string())
LOG["추세_1천명당kWh"] = {k: [float(first_.get(k, np.nan)), float(last_.get(k, np.nan))] for k in TSG}

In [ ]:
# [16] 매칭 쌍 — 조건이 비슷한데 공백·충전이 갈리는 곳 (사례 선정용, 인과 추정 아님)
rng_ = np.random.RandomState(0)
Xc = np.c_[np.log(POP.clip(min=1)), np.nan_to_num(dong["고령비율"].values), np.log(dong["면적km2"].fillna(1).values.clip(min=0.01))]
okc = np.isfinite(Xc).all(axis=1)
try:
    CI = np.linalg.inv(np.cov(Xc[okc], rowvar=False))
except np.linalg.LinAlgError:
    CI = np.linalg.pinv(np.cov(Xc[okc], rowvar=False))
treated = np.where(gnow["strict"] & okc)[0]
poold = np.where(~gnow["centroid"] & okc)[0]
dn_flag = (dong["인구감소지역"] == "Y").values
mrows = []
for i in treated:
    cand = poold[dn_flag[poold] == dn_flag[i]]           # 인구감소지역 여부 정확일치
    if len(cand) == 0:
        cand = poold
    diff = Xc[cand] - Xc[i]
    dm = np.sqrt(np.einsum("ij,jk,ik->i", diff, CI, diff))
    j = cand[int(np.argmin(dm))]
    mrows.append(dict(공백행정동=dong["행정동명"].iloc[i], 비교행정동=dong["행정동명"].iloc[j], 마할라노비스=round(float(dm.min()), 2),
                      인구_공백=int(POP[i]), 인구_비교=int(POP[j]), 고령비율_공백=round(float(dong["고령비율"].iloc[i]), 3), 고령비율_비교=round(float(dong["고령비율"].iloc[j]), 3),
                      kWh천명당_공백=round(float(p12[i]), 3), kWh천명당_비교=round(float(p12[j]), 3), 충전기수_공백=int(cnt[i]), 충전기수_비교=int(cnt[j]),
                      자가충전비중_공백=nb_share[i], 자가충전비중_비교=nb_share[j]))
MT = pd.DataFrame(mrows)
if len(MT):
    d_ = (MT["kWh천명당_공백"] - MT["kWh천명당_비교"]).values
    obs = d_.mean()
    signs = rng_.choice([-1, 1], size=(5000, len(d_)))
    pperm = float((np.abs((signs * d_).mean(axis=1)) >= abs(obs) - 1e-12).mean())
    print("[행정동 매칭] 공백 %d곳 ↔ 조건이 비슷한 비공백 %d곳" % (len(MT), len(MT)))
    print("  최근12개월 1천명당 월 kWh 평균: 공백 %.3f vs 비교 %.3f (차이 %.3f, 부호뒤집기 순열검정 p=%.4f)" % (MT["kWh천명당_공백"].mean(), MT["kWh천명당_비교"].mean(), obs, pperm))
    MT.to_csv(os.path.join(OUT_DIR, "내부용", "11_매칭_행정동.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame([dict(쌍수=len(MT), 공백_kWh천명당=MT["kWh천명당_공백"].mean(), 비교_kWh천명당=MT["kWh천명당_비교"].mean(), 평균차이=obs, 순열검정_p=pperm)]).round(4).to_csv(
        os.path.join(OUT_DIR, "반출후보", "11_매칭_요약.csv"), index=False, encoding="utf-8-sig")
    LOG["매칭_행정동"] = dict(쌍수=len(MT), 공백=round(float(MT["kWh천명당_공백"].mean()), 3), 비교=round(float(MT["kWh천명당_비교"].mean()), 3), p=round(pperm, 4))
else:
    print("[행정동 매칭] 엄격 공백 행정동이 없어 생략")

# 시군구 단위: 이중공백 노출이 큰 시군구 vs 조건이 비슷한데 이중공백이 없는 시군구
sg = sggm.copy()
for c in ["총인구", "인구65", "면적km2", "고령비율", "연평균증감률", "인구밀도"]:
    sg[c] = pd.to_numeric(sg[c], errors="coerce")
pe = EXP.groupby("시군구명").agg(노출=("노출_인구개월_strict", "sum"), 현재이중공백인구=("인구", lambda x: x[EXP.loc[x.index, "이중공백_현재_strict"]].sum()))
sg = sg.merge(pe, left_on="시군구명", right_index=True, how="left").fillna({"노출": 0, "현재이중공백인구": 0})
kw_sg = pd.Series(kw12, index=dong.index).groupby(dong["시군구명"]).sum()
sg["kWh천명당_월"] = sg["시군구명"].map(kw_sg) / 12 / sg["총인구"] * 1000
sg["이중공백인구비율"] = sg["현재이중공백인구"] / sg["총인구"]
Xs = np.c_[np.log(sg["총인구"].clip(lower=1)), sg["고령비율"].fillna(sg["고령비율"].median()), sg["연평균증감률"].fillna(0), np.log(sg["인구밀도"].clip(lower=0.01))]
CIs = np.linalg.pinv(np.cov(Xs, rowvar=False))
tops = sg.sort_values("노출", ascending=False).head(8)
srows = []
for i, r in tops.iterrows():
    if r["노출"] <= 0:
        continue
    pool = sg[(sg["노출"] == 0) & (sg["현재이중공백인구"] == 0) & (sg["인구감소지역구분"] == r["인구감소지역구분"]) & (sg.index != i)]   # 이중공백 이력이 전혀 없는 시군구만
    if len(pool) == 0:
        continue
    diff = Xs[pool.index] - Xs[i]
    dm = np.sqrt(np.einsum("ij,jk,ik->i", diff, CIs, diff))
    j = pool.index[int(np.argmin(dm))]
    srows.append(dict(대상=r["시군구명"], 비교=sg.loc[j, "시군구명"], 마할라노비스=round(float(dm.min()), 2), 대상_이중공백인구비율=round(r["이중공백인구비율"], 4), 비교_이중공백인구비율=0.0,
                      대상_kWh천명당=round(r["kWh천명당_월"], 3), 비교_kWh천명당=round(sg.loc[j, "kWh천명당_월"], 3), 대상_인구=int(r["총인구"]), 비교_인구=int(sg.loc[j, "총인구"]),
                      대상_고령비율=round(r["고령비율"], 3), 비교_고령비율=round(sg.loc[j, "고령비율"], 3), 대상_인구밀도=round(r["인구밀도"], 1), 비교_인구밀도=round(sg.loc[j, "인구밀도"], 1)))
MS = pd.DataFrame(srows)
if len(MS):
    print("\n[시군구 매칭]"); print(MS[["대상", "비교", "마할라노비스", "대상_이중공백인구비율", "대상_kWh천명당", "비교_kWh천명당"]].to_string(index=False))
    MS.to_csv(os.path.join(OUT_DIR, "반출후보", "11_매칭_시군구.csv"), index=False, encoding="utf-8-sig")

In [ ]:
# [17] 민감도 요약 — 공백 정의(2) × 거리 기준(3) × 충전 기준(3)
rows = []
for d in DEFS:
    for km in GAP_KMS:
        gg_ = GAP[(d, km)]
        gn = gg_["above"][:, LATEST]
        for q in CHARGE_QS:
            lw = p12 <= THR12[q]
            on = onset_table(d, km, q)
            ev_ = on[on["상태"] == "event"]
            lt_ = on[on["상태"] == "left_truncated"]
            def med(df):
                s = df[df["기간_개월"].notna()]
                return km_median(km_table(s["기간_개월"], s["사건"], s["진입"])) if len(s) and s["사건"].sum() > 0 else np.nan
            rows.append(dict(공백정의=d, 거리기준km=km, 충전기준분위=q, 공백_현재_행정동=int(gn.sum()), 이중공백_현재_행정동=int((gn & lw).sum()), 이중공백_현재_인구=int(POP[gn & lw].sum()),
                             공백시작_관측=int((gg_["status"] == "event").sum()), 관측이전부터_공백=int((gg_["status"] == "left_truncated").sum()),
                             시차_중앙값_event=med(ev_), 시차_중앙값_LT=med(lt_)))
SENS = pd.DataFrame(rows)
print(SENS.to_string(index=False))
mask_df(SENS, ["공백_현재_행정동", "이중공백_현재_행정동", "공백시작_관측", "관측이전부터_공백"]).to_csv(os.path.join(OUT_DIR, "반출후보", "12_민감도_요약.csv"), index=False, encoding="utf-8-sig")
LOG["민감도_행수"] = len(SENS)

In [ ]:
# [18] 그림 저장 (matplotlib 이 있을 때만) — 결과/그림 폴더
if OPT.get("matplotlib"):
    import logging
    logging.getLogger("matplotlib").setLevel(logging.ERROR)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    names_ = {f.name for f in fm.fontManager.ttflist}
    for fam in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
        if fam in names_:
            plt.rcParams["font.family"] = fam
            break
    plt.rcParams["axes.unicode_minus"] = False
    FIG = os.path.join(OUT_DIR, "그림")
    # 1) 로렌츠
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.plot([0, 1], [0, 1], "--", color="gray", label="완전 평등")
    ax.plot(LZ["누적행정동비율_개소수"], LZ["누적비율_개소수"], label="충전기 개소수(지니 %.2f)" % GINI.iloc[0]["지니_행정동동일가중"])
    ax.plot(LZ["누적행정동비율_kWh"], LZ["누적비율_kWh"], label="실제 kWh(지니 %.2f)" % GINI.iloc[1]["지니_행정동동일가중"])
    ax.set_xlabel("누적 행정동 비율(적은 순)"); ax.set_ylabel("누적 비율"); ax.legend(); ax.set_title("충전기 개소수 vs 실제 kWh")
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "01_로렌츠.png"), dpi=150); plt.close(fig)
    # 2) 달력 시간 KM
    if CAL:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for lab, g_ in pd.concat(CAL, ignore_index=True).groupby("집단"):
            ax.step(np.r_[0, g_["개월"]], np.r_[1, g_["생존확률"]], where="post", label=lab)
        ax.set_xlabel("한전 자료 시작 후 개월"); ax.set_ylabel("아직 충전 실사용 도달 못한 비율"); ax.legend(); ax.set_title("충전 실사용 도달까지(Kaplan-Meier)")
        fig.tight_layout(); fig.savefig(os.path.join(FIG, "02_충전도달_KM.png"), dpi=150); plt.close(fig)
    # 3) 거리 × 충전량 산점도
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(np.clip(D_[:, LATEST], 0.01, None), np.clip(p12, 1e-3, None), s=6, alpha=0.35, color="gray")
    mk_ = gnow["strict"]
    ax.scatter(np.clip(D_[mk_, LATEST], 0.01, None), np.clip(p12[mk_], 1e-3, None), s=24, color="crimson", label="엄격 공백(현재)")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.axvline(PRIMARY_GAP_KM, color="k", ls=":"); ax.axhline(THR12[PRIMARY_CHARGE_Q], color="k", ls=":")
    ax.set_xlabel("최근접 주유소 거리(km, 중심점)"); ax.set_ylabel("최근 12개월 1천명당 월 kWh"); ax.legend(); ax.set_title("4분면 (점선 = 기준)")
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "03_4분면_산점도.png"), dpi=150); plt.close(fig)
    # 4) 월별 추세
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for lab, g_ in TSR.groupby("집단"):
        ax.plot(pd.to_datetime(g_["월"] + "-01"), g_["1천명당_kWh"], label=lab)
    ax.set_ylabel("인구 1천명당 월 kWh"); ax.legend(fontsize=8); ax.set_title("월별 충전량 추세")
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "04_월별_추세.png"), dpi=150); plt.close(fig)
    print("그림 4개 저장:", FIG)
else:
    print("(matplotlib 이 없어 그림은 생략 — 표(CSV)는 모두 저장됩니다)")

In [ ]:
# [19] 결과 정리 — 반출 후보/내부용 구분, 결과물 목록, 실행 요약
MAN = [
 ("반출후보/01_품질점검.csv", "전국", "한전 자료 품질(연결률·결측·마스킹·기간)"),
 ("반출후보/03_지니계수.csv", "전국", "충전기 개수 vs kWh 지니계수(동일가중/인구가중)"),
 ("반출후보/03_지니계수_연도별.csv", "전국", "연도별 kWh 지니계수"),
 ("반출후보/03_로렌츠곡선_점.csv", "전국", "로렌츠 곡선 좌표(101점)"),
 ("반출후보/05_4분면_요약.csv", "전국", "공백×충전 4분면별 행정동 수·인구(소규모 셀 가림)"),
 ("반출후보/06_전환시차_요약.csv", "전국", "공백 시작 후 충전 시작까지 시차 요약(그룹별)"),
 ("반출후보/06_전환시차_KM곡선.csv", "전국", "시차 Kaplan-Meier 곡선 좌표"),
 ("반출후보/06_달력시간_생존_요약.csv", "전국", "공백 vs 그 외의 충전 도달 속도 비교(log-rank)"),
 ("반출후보/06_달력시간_KM곡선.csv", "전국", "달력 시간 Kaplan-Meier 곡선 좌표"),
 ("반출후보/06_Cox_결과.csv", "전국", "Cox 위험비(statsmodels 가 있을 때)"),
 ("반출후보/07_이중공백_노출_지역별.csv", "시도·시군구", "이중공백 노출 인구·개월(지역별 합계)"),
 ("반출후보/08_정책유형_요약.csv", "전국", "정책 유형별 행정동 수·인구"),
 ("반출후보/09_자가충전_야간충전.csv", "집단별", "비사업자(자가충전) 비중·야간 비중"),
 ("반출후보/10_월별_추세.csv", "집단별", "월별 1천명당 kWh 추세"),
 ("반출후보/11_매칭_요약.csv", "전국", "매칭 쌍 평균 비교"),
 ("반출후보/11_매칭_시군구.csv", "시군구", "시군구 매칭 쌍(사례)"),
 ("반출후보/12_민감도_요약.csv", "전국", "공백정의×거리×충전기준 민감도"),
 ("내부용/00_지역연결표.csv", "한전 지역 단위", "한전 지역 → 행정동 연결표(반출 비권장)"),
 ("내부용/02_행정동_월별_충전량.csv", "행정동×월", "행정동별 월 kWh(반출 비권장: 소규모 지역 식별 우려)"),
 ("내부용/04_가동률_유령충전기.csv", "행정동", "행정동별 가동률·유령 후보(반출 비권장)"),
 ("내부용/05_행정동_공백_충전_분류.csv", "행정동", "행정동별 공백·충전·4분면(반출 비권장)"),
 ("내부용/06_전환시차_개별_행정동.csv", "행정동", "행정동별 공백 시작·충전 시작·시차(반출 비권장)"),
 ("내부용/07_이중공백_노출_행정동.csv", "행정동", "행정동별 이중공백 개월·노출(반출 비권장)"),
 ("내부용/08_정책유형_행정동.csv", "행정동", "행정동별 정책 유형(반출 비권장)"),
 ("내부용/11_매칭_행정동.csv", "행정동", "행정동 매칭 쌍(반출 비권장)"),
 ("그림/01_로렌츠.png ~ 04_월별_추세.png", "전국", "그림 4개"),
]
pd.DataFrame(MAN, columns=["파일", "단위", "내용"]).to_csv(os.path.join(OUT_DIR, "00_결과물_목록.csv"), index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "반출후보", "README_반출전_확인.txt"), "w", encoding="utf-8") as f:
    f.write("반출 전 확인 사항\n1) 이 폴더(반출후보)의 표는 전국·시도·시군구·집단 단위 집계입니다. 행정동 단위 표는 '내부용' 폴더에 있으며 반출을 권하지 않습니다.\n"
            "2) 건수가 %d 미만인 칸은 '<%d' 로 가려져 있습니다(셀 1 의 MIN_CELL). 안심구역 반출 규정의 기준값으로 조정하세요.\n3) 반출 심사 담당자 안내에 따라 파일 목록을 제출하세요(00_결과물_목록.csv).\n" % (MIN_CELL, MIN_CELL))
LOG["설정"] = dict(PRIMARY_GAP_KM=PRIMARY_GAP_KM, PRIMARY_CHARGE_Q=PRIMARY_CHARGE_Q, RUN_MONTHS=RUN_MONTHS, GHOST_Q=GHOST_Q, MULTI_RULE=MULTI_RULE, MIN_CELL=MIN_CELL,
                   HOUR_LABEL_END=HOUR_LABEL_END, NIGHT_IDX=NIGHT_IDX, UNIT_FACTOR=UNIT_FACTOR)
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if (o != o) else float(o)
    return o
with open(os.path.join(OUT_DIR, "반출후보", "00_실행요약.json"), "w", encoding="utf-8") as f:
    json.dump(_clean(LOG), f, ensure_ascii=False, indent=1)
print("저장 완료 →", os.path.abspath(OUT_DIR))
for sub in ["반출후보", "내부용", "그림"]:
    print("  %s: %s" % (sub, sorted(os.listdir(os.path.join(OUT_DIR, sub)))))

In [ ]:
# [20] 자체 점검 — 결과가 정상인지 마지막 확인
problems = []
if LOG["지역연결률_kWh가중"] < 0.9:
    problems.append("지역 연결률이 90%% 미만(%.1f%%) — 셀 6 의 미연결 목록 확인" % (100 * LOG["지역연결률_kWh가중"]))
if abs(KW.values.sum() - M["kwh"].sum() * LOG["지역연결률_kWh가중"]) > 0.02 * max(M["kwh"].sum(), 1):
    problems.append("행정동 반영 kWh 와 연결률 기준 기대치가 2%% 이상 다름 — 법정동 중복/누락 확인")
if (KW.values < 0).any():
    problems.append("음수 kWh 가 있음 — 정산/오류 자료 여부 확인")
if not np.isfinite(p12).all():
    problems.append("1천명당 kWh 에 비정상 값(NaN/inf) — 인구 0 인 행정동 확인")
if (dong["한전자료있음"].mean() < 0.5):
    problems.append("한전 자료가 붙은 행정동이 절반 미만 — 지역 연결 또는 자료 범위 확인")
need_files = ["반출후보/03_지니계수.csv", "반출후보/05_4분면_요약.csv", "반출후보/06_전환시차_요약.csv", "반출후보/07_이중공백_노출_지역별.csv", "내부용/08_정책유형_행정동.csv"]
for nf in need_files:
    if not os.path.exists(os.path.join(OUT_DIR, nf)):
        problems.append("결과 파일 누락: " + nf)
if problems:
    print("점검 결과: 확인이 필요한 항목 %d개" % len(problems))
    for p_ in problems:
        print("  -", p_)
else:
    print("점검 통과: 지역 연결률 %.1f%%, kWh 보존, 결과 파일 모두 생성" % (100 * LOG["지역연결률_kWh가중"]))